In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"

print(DATA_ROOT)

c:\Users\tkdoa\Desktop\lab2\data


In [16]:
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

SEED = 1234
GENERATED_METADATA_PATH = ARTIFACT_DIR / f"lab2_faces_metadata.csv"

SPLITS = ("train", "val", "test")
LABELS = ("cat", "dog")
IMAGE_EXTENSIONS = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")

print("DATA_ROOT:", DATA_ROOT)

DATA_ROOT: c:\Users\tkdoa\Desktop\lab2\data


In [18]:
def list_image_paths_for_group(data_root: Path, split: str, label: str):
    folder = data_root / split / label
    paths = []
    
    for ext in IMAGE_EXTENSIONS:
        paths.extend(folder.glob(ext))
    
    return paths

In [19]:
def inspect_image_file(path: Path):
    with Image.open(path) as img:
        img = img.convert("RGB")
        width, height = img.size
        
        arr = np.array(img).astype(np.float32) / 255.0
        mean_intensity = arr.mean()
    
    return width, height, float(mean_intensity)

In [21]:
def make_metadata_row(path: Path, data_root: Path, split: str, label: str):
    width, height, mean_intensity = inspect_image_file(path)
    
    return {
        "filepath": str(path.relative_to(data_root)),
        "label": label,
        "split": split,
        "width": width,
        "height": height,
        "mean_intensity": mean_intensity,
    }

In [22]:
def build_metadata_from_folders(data_root: Path):
    rows = []
    
    for split in SPLITS:
        for label in LABELS:
            paths = list_image_paths_for_group(data_root, split, label)
            
            for p in paths:
                row = make_metadata_row(p, data_root, split, label)
                rows.append(row)
    
    df = pd.DataFrame(rows)
    
    df = df.sort_values(["split", "label", "filepath"]).reset_index(drop=True)
    
    return df

In [25]:
folder_df = build_metadata_from_folders(DATA_ROOT)

print("Shape:", folder_df.shape)
folder_df.head()

Shape: (19, 6)


,filepath,label,split,width,height,mean_intensity
0,test\cat\OIP.jpg,cat,test,270,180,0.268178
1,test\cat\oo.jpg,cat,test,270,180,0.384233
2,test\dog\OIP.jpg,dog,test,288,180,0.589502
3,test\dog\dog.jpg,dog,test,240,180,0.403101
4,train\cat\4936910_cute-cats-wallpapers-5____by...,cat,train,1680,1050,0.581512


In [28]:
folder_df.to_csv(GENERATED_METADATA_PATH, index=False)

In [30]:
def load_metadata_table(csv_path: Path):
    return pd.read_csv(csv_path)

In [31]:
df = load_metadata_table(GENERATED_METADATA_PATH)
print(df.shape)
df.head()

(19, 6)


,filepath,label,split,width,height,mean_intensity
0,test\cat\OIP.jpg,cat,test,270,180,0.268178
1,test\cat\oo.jpg,cat,test,270,180,0.384233
2,test\dog\OIP.jpg,dog,test,288,180,0.589502
3,test\dog\dog.jpg,dog,test,240,180,0.403101
4,train\cat\4936910_cute-cats-wallpapers-5____by...,cat,train,1680,1050,0.581512


In [33]:
def summarize_metadata(frame: pd.DataFrame):
    return {
        "rows": len(frame),
        "columns": list(frame.columns),
        "class_counts": frame["label"].value_counts(),
        "split_counts": frame["split"].value_counts(),
    }

In [35]:
summary = summarize_metadata(df)

print("Rows:", summary["rows"])
print("Columns:", summary["columns"])

print("\nClass counts:")
print(summary["class_counts"])

print("\nSplit counts:")
print(summary["split_counts"])

Rows: 19
Columns: ['filepath', 'label', 'split', 'width', 'height', 'mean_intensity']

Class counts:
label
cat    10
dog     9
Name: count, dtype: int64

Split counts:
split
train    11
test      4
val       4
Name: count, dtype: int64


In [37]:
def build_label_split_table(frame: pd.DataFrame):
    return frame.groupby(["label", "split"]).size().unstack(fill_value=0)

In [41]:
label_split_table = build_label_split_table(df)
label_split_table

split,test,train,val
label,,,
cat,2,6,2
dog,2,5,2


In [42]:
def audit_metadata(frame: pd.DataFrame):
    return {
        "missing_values": frame.isna().sum().to_dict(),
        "duplicate_filepaths": frame["filepath"].duplicated().sum(),
        "bad_labels": list(set(frame["label"]) - set(LABELS)),
        "non_positive_sizes": ((frame["width"] <= 0) | (frame["height"] <= 0)).sum(),
    }

In [43]:
audit_report = audit_metadata(df)
print(audit_report)

{'missing_values': {'filepath': 0, 'label': 0, 'split': 0, 'width': 0, 'height': 0, 'mean_intensity': 0}, 'duplicate_filepaths': np.int64(0), 'bad_labels': [], 'non_positive_sizes': np.int64(0)}


In [45]:
def add_analysis_columns(frame: pd.DataFrame):
    df = frame.copy()

    # 1. pixel_count
    df["pixel_count"] = df["width"] * df["height"]

    # 2. aspect_ratio
    df["aspect_ratio"] = df["width"] / df["height"]

    # 3. brightness_band (chia 4 mức)
    df["brightness_band"] = pd.qcut(
        df["mean_intensity"],
        4,
        labels=["darkest", "dim", "bright", "brightest"]
    )

    # 4. size_bucket
    ref = 64 * 64

    def size_bucket(x):
        if x < ref:
            return "small"
        elif x == ref:
            return "medium"
        else:
            return "large"

    df["size_bucket"] = df["pixel_count"].apply(size_bucket)

    return df

In [47]:
analysis_df = add_analysis_columns(df)
analysis_df.head()

,filepath,label,split,width,height,mean_intensity,pixel_count,aspect_ratio,brightness_band,size_bucket
0,test\cat\OIP.jpg,cat,test,270,180,0.268178,48600,1.500000,darkest,large
1,test\cat\oo.jpg,cat,test,270,180,0.384233,48600,1.500000,dim,large
2,test\dog\OIP.jpg,dog,test,288,180,0.589502,51840,1.600000,brightest,large
3,test\dog\dog.jpg,dog,test,240,180,0.403101,43200,1.333333,dim,large
4,train\cat\4936910_cute-cats-wallpapers-5____by...,cat,train,1680,1050,0.581512,1764000,1.600000,brightest,large


In [51]:
def build_split_characteristics_table(frame: pd.DataFrame):
    return (
        frame.groupby("split")
        .agg(
            avg_width=("width", "mean"),
            avg_height=("height", "mean"),
            avg_pixel_count=("pixel_count", "mean"),
            avg_mean_intensity=("mean_intensity", "mean"),
        )
        .reset_index()
    )

In [52]:
split_characteristics = build_split_characteristics_table(analysis_df)
split_characteristics

,split,avg_width,avg_height,avg_pixel_count,avg_mean_intensity
0,test,267.000000,180.000000,4.806000e+04,0.411254
1,train,1303.545455,856.818182,1.385578e+06,0.449402
2,val,277.500000,180.000000,4.995000e+04,0.435390


In [53]:
def sample_balanced_by_split_and_label(frame, n_per_group, seed):
    return (
        frame.groupby(["split", "label"], group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), n_per_group), random_state=seed))
        .reset_index(drop=True)
    )

In [54]:
sample_size_per_group = 5

sampled_df = sample_balanced_by_split_and_label(
    analysis_df,
    n_per_group=sample_size_per_group,
    seed=SEED
)

print("Shape:", sampled_df.shape)
sampled_df.head()

Shape: (18, 10)


C:\Users\tkdoa\AppData\Local\Temp\ipykernel_7556\3493095862.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), n_per_group), random_state=seed))


,filepath,label,split,width,height,mean_intensity,pixel_count,aspect_ratio,brightness_band,size_bucket
0,test\cat\OIP.jpg,cat,test,270,180,0.268178,48600,1.500000,darkest,large
1,test\cat\oo.jpg,cat,test,270,180,0.384233,48600,1.500000,dim,large
2,test\dog\OIP.jpg,dog,test,288,180,0.589502,51840,1.600000,brightest,large
3,test\dog\dog.jpg,dog,test,240,180,0.403101,43200,1.333333,dim,large
4,train\cat\Meo-1.jpg,cat,train,1933,1088,0.453210,2103104,1.776654,bright,large


In [55]:
sampled_df.groupby(["split", "label"]).size().unstack(fill_value=0)

label,cat,dog
split,,
test,2,2
train,5,5
val,2,2
